# Primetrade.ai — Trader Performance vs Market Sentiment
**Author:** Deep Kumar | **Dataset:** Hyperliquid + Bitcoin Fear & Greed Index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

## 1. Load & Explore Data

In [ ]:
# Load datasets
df = pd.read_csv('historical_data.csv')
fg = pd.read_csv('fear_greed_index.csv')

print('Trader Data Shape:', df.shape)
print('Fear/Greed Shape:', fg.shape)
df.head()

In [ ]:
# Explore Fear/Greed
print('Sentiment distribution:')
print(fg['classification'].value_counts())
fg.head()

## 2. Data Cleaning & Merging

In [ ]:
# Parse dates
df['date'] = pd.to_datetime(df['Timestamp IST'], dayfirst=True).dt.date.astype(str)
fg['date'] = pd.to_datetime(fg['date']).dt.date.astype(str)

# Merge
merged = df.merge(fg[['date', 'value', 'classification']], on='date', how='inner')
merged.rename(columns={'value': 'sentiment_score', 'classification': 'sentiment'}, inplace=True)

print(f'Merged rows: {len(merged):,}')
print(f'Date range: {merged["date"].min()} → {merged["date"].max()}')
merged[['date', 'Coin', 'Closed PnL', 'sentiment', 'sentiment_score']].head()

## 3. Sentiment Analysis

In [ ]:
SENTIMENT_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
SENTIMENT_COLORS = {
    'Extreme Fear': '#f85149', 'Fear': '#ff7b72',
    'Neutral': '#e3b341', 'Greed': '#56d364', 'Extreme Greed': '#3fb950'
}

sent = merged.groupby('sentiment').agg(
    total_pnl  = ('Closed PnL', 'sum'),
    avg_pnl    = ('Closed PnL', 'mean'),
    trades     = ('Closed PnL', 'count'),
    win_trades = ('Closed PnL', lambda x: (x > 0).sum()),
    vol_usd    = ('Size USD', 'sum'),
).reset_index()
sent['win_rate'] = sent['win_trades'] / sent['trades'] * 100
sent = sent.set_index('sentiment').reindex(SENTIMENT_ORDER).reset_index()
sent

In [ ]:
# Chart: Avg PnL by Sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [SENTIMENT_COLORS[s] for s in sent['sentiment']]
axes[0].bar(sent['sentiment'], sent['avg_pnl'], color=colors)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('Avg PnL by Sentiment')
axes[0].set_ylabel('Avg Closed PnL (USD)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(sent['sentiment'], sent['win_rate'], color=colors)
axes[1].axhline(50, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title('Win Rate by Sentiment')
axes[1].set_ylabel('Win Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
daily = merged.groupby(['date', 'sentiment', 'sentiment_score']).agg(
    total_pnl = ('Closed PnL', 'sum'),
    trades    = ('Closed PnL', 'count'),
).reset_index()

slope, intercept, r, p, _ = stats.linregress(daily['sentiment_score'], daily['total_pnl'])
print(f'Pearson r = {r:.3f}, p-value = {p:.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
for s in SENTIMENT_ORDER:
    sub = daily[daily['sentiment'] == s]
    ax.scatter(sub['sentiment_score'], sub['total_pnl'], color=SENTIMENT_COLORS[s], alpha=0.7, s=40, label=s)
x_line = np.linspace(daily['sentiment_score'].min(), daily['sentiment_score'].max(), 100)
ax.plot(x_line, slope*x_line + intercept, 'b--', label=f'Trend (r={r:.2f})')
ax.axhline(0, color='gray', linewidth=0.6)
ax.set_title('Fear & Greed Score vs Daily PnL')
ax.set_xlabel('Fear & Greed Score')
ax.set_ylabel('Daily Total PnL (USD)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Coin × Sentiment Heatmap

In [ ]:
top_coins = merged.groupby('Coin')['Closed PnL'].count().nlargest(12).index
heat_data = merged[merged['Coin'].isin(top_coins)].groupby(['Coin', 'sentiment'])['Closed PnL'].mean().unstack()
heat_data = heat_data.reindex(columns=SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(heat_data, ax=ax, cmap='RdYlGn', center=0, fmt='.0f', annot=True,
            linewidths=0.4)
ax.set_title('Avg PnL by Coin × Sentiment (Top 12 Coins)')
plt.tight_layout()
plt.show()

## 6. Key Insights & Strategy Recommendations

1. **Extreme Greed = Best performance** — Highest avg PnL ($67.89) and win rate (46.5%)\n2. **Fear ≠ Bad** — Fear periods still deliver $54.29 avg PnL; skilled traders buy dips\n3. **Avoid Neutral markets** — Lowest avg PnL ($34.31), choppy conditions\n4. **BUY dominates** — Long trades outperform shorts in all sentiment categories during bull cycles\n5. **Sentiment score correlation is weak** (r=-0.083) — use categorical signals, not raw score\n